In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

# 1. Load MNIST dataset 
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_raw, y_raw = mnist.data.astype(np.float32), mnist.target.astype(int)

# 2. Preprocess: Normalize pixels to [0, 1] 
X_raw /= 255.0

# 3. Binary Labels: 1 if even (0,2,4,6,8), 0 if odd (1,3,5,7,9) 
y_binary = (y_raw % 2 == 0).astype(np.float32).reshape(-1, 1)

# 4. Train/Test Split: First 60k for training, last 10k for testing
X_train, X_test = X_raw[:60000], X_raw[60000:]
y_train, y_test = y_binary[:60000], y_binary[60000:]

In [2]:
def sigmoid(z):
    z = np.clip(z, -500, 500) # Prevent overflow 
    return 1 / (1 + np.exp(-z)) # Standard sigmoid formula

def forward(X, W1, b1, W2, b2):
    # Step 1: Hidden Layer (Linear + Tanh)
    Z1 = np.dot(X, W1) + b1 # (n, 784) * (784, 64) + (64,) -> (n, 64) 
    A1 = np.tanh(Z1)        # Apply non-linear activation 
    
    # Step 2: Output Layer (Linear + Sigmoid) [cite: 125]
    Z2 = np.dot(A1, W2) + b2 # (n, 64) * (64, 1) + (1,) -> (n, 1)
    y_hat = sigmoid(Z2)      # Output probability in [0, 1] 
    
    cache = (Z1, A1, Z2) # Store for backpropagation 
    return y_hat, cache

def compute_loss(y_true, y_hat):
    eps = 1e-15 # Constant to avoid log(0)
    y_hat = np.clip(y_hat, eps, 1 - eps) # Clip values 
    # Calculate average loss across the batch 
    return -np.mean(y_true * np.log(y_hat) + (1 - y_true) * np.log(1 - y_hat))

def backward(X, y, y_hat, cache, W2):
    n = X.shape[0] # Number of samples
    Z1, A1, Z2 = cache # Retrieve intermediate values
    
    # 1. Output Layer Error 
    delta2 = y_hat - y # Difference between prediction and truth (n, 1)
    
    # 2. Output Layer Gradients 
    dW2 = (1/n) * np.dot(A1.T, delta2) # Gradient for W2 (64, 1)
    db2 = (1/n) * np.sum(delta2, axis=0) # Gradient for b2 (1,)
    
    # 3. Hidden Layer Error 
    # Backpropagate error through W2 and tanh derivative (1 - A1^2) 
    delta1 = np.dot(delta2, W2.T) * (1 - np.power(A1, 2)) # (n, 64)
    
    # 4. Hidden Layer Gradients 
    dW1 = (1/n) * np.dot(X.T, delta1) # Gradient for W1 (784, 64)
    db1 = (1/n) * np.sum(delta1, axis=0) # Gradient for b1 (64,)
    
    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads

def train(X, y, hidden_size=64, learning_rate=0.1, iterations=1000):
    # Initialize parameters with small random values
    W1 = np.random.randn(784, hidden_size).astype(np.float32) * 0.01
    b1 = np.zeros(hidden_size, dtype=np.float32)
    W2 = np.random.randn(hidden_size, 1).astype(np.float32) * 0.01
    b2 = np.zeros(1, dtype=np.float32)
    
    losses = []
    
    for i in range(iterations):
        # Forward pass
        y_hat, cache = forward(X, W1, b1, W2, b2)
        
        # Compute and record loss
        curr_loss = compute_loss(y, y_hat)
        losses.append(curr_loss)
        
        # Backward pass
        grads = backward(X, y, y_hat, cache, W2)
        
        # Update parameters 
        W1 -= learning_rate * grads['dW1']
        b1 -= learning_rate * grads['db1']
        W2 -= learning_rate * grads['dW2']
        b2 -= learning_rate * grads['db2']
        
        # Optional: Print progress every 100 iterations
        if i % 100 == 0:
            print(f"Iteration {i}: Loss {curr_loss:.4f}")
            
    return W1, b1, W2, b2, losses

In [3]:
# Run training with standard hyperparameters 
W1_final, b1_final, W2_final, b2_final, train_losses = train(
    X_train, y_train, hidden_size=64, learning_rate=0.5, iterations=1000
)

# Test the model
y_test_hat, _ = forward(X_test, W1_final, b1_final, W2_final, b2_final)
predictions = (y_test_hat > 0.5).astype(np.float32) # Convert to 0/1 
accuracy = np.mean(predictions == y_test)
print(f"Final Test Accuracy: {accuracy * 100:.2f}%") # Expected 90-95% 

Iteration 0: Loss 0.6927
Iteration 100: Loss 0.3101
Iteration 200: Loss 0.2591
Iteration 300: Loss 0.1965
Iteration 400: Loss 0.1442
Iteration 500: Loss 0.1220
Iteration 600: Loss 0.1112
Iteration 700: Loss 0.0949
Iteration 800: Loss 0.0881
Iteration 900: Loss 0.0800
Final Test Accuracy: 97.35%
